# 05 — Ensemble + Corrección Index_D
**Entrega 5.** Combina baseline, ARIMA, LightGBM y LSTM con pesos optimizados.
Aplica corrección lineal a Index_D usando su índice fuente identificado en EDA.

In [13]:
import sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
from utils import load_data, compute_rmse, make_submission, train_val_split, find_ghost_source, INDEX_COLS

data       = load_data()
train_full = data["train_indices"][INDEX_COLS]
test_dates = data["test_dates"].index
train, val = train_val_split(train_full, val_size=252)


## Cargar predicciones individuales de validación

In [14]:
SUBMISSIONS = "submissions"

def load_sub(fname, dates):
    df = pd.read_csv(f"{SUBMISSIONS}/{fname}", parse_dates=[0], index_col=0)
    return df.reindex(dates)[INDEX_COLS]

# Asegúrate de haber corrido los notebooks anteriores con datos de validación.
# Aquí cargamos los CSVs de test y re-evaluamos sobre val localmente.

# ----- Baseline -----
def rolling_mean(series_df, dates, window=20):
    m = series_df.tail(window).mean()
    return pd.DataFrame({c: [m[c]]*len(dates) for c in INDEX_COLS}, index=dates)

pred_baseline = rolling_mean(train, val.index, 20)
print(f"Baseline RMSE : {compute_rmse(val, pred_baseline):,.2f}")


Baseline RMSE : 3,507.41


## Re-generar predicciones val de LightGBM

In [15]:
# Importamos las funciones directamente del notebook 03
from utils import create_lag_features, add_calendar_features, add_log_returns

macro   = data.get("train_macro_factors")
network = data.get("train_network_metrics")
macro_tr = macro.iloc[:-252] if macro is not None else None
net_tr   = network.iloc[:-252] if network is not None else None

LAGS = (1, 2, 3, 5, 10, 20, 60); WINDOWS = (5, 10, 20, 60)

def build_features(indices, macro=None, network=None):
    df = indices[INDEX_COLS].copy()
    df = add_log_returns(df)
    df = create_lag_features(df, lags=LAGS, windows=WINDOWS)
    df = add_calendar_features(df)
    if macro   is not None: df = pd.concat([df, macro.reindex(df.index).ffill()],   axis=1)
    if network is not None: df = pd.concat([df, network.reindex(df.index).ffill()], axis=1)
    return df

def prepare_xy(feats, targets):
    rename = {c: f"__t_{c}" for c in INDEX_COLS}
    combined = pd.concat([feats, targets.rename(columns=rename)], axis=1).dropna()
    tc = list(rename.values())
    return combined.drop(columns=tc).values, combined[tc].values, combined.index

def train_lgbm(X, y, n=500):
    try:
        import lightgbm as lgb
        ms = []
        for i in range(6):
            m = lgb.LGBMRegressor(n_estimators=n, learning_rate=0.05,
                                  num_leaves=63, subsample=0.8,
                                  colsample_bytree=0.8, verbose=-1)
            m.fit(X, y[:, i]); ms.append(m)
        return ms
    except ImportError:
        from sklearn.ensemble import GradientBoostingRegressor
        return [GradientBoostingRegressor(n_estimators=200).fit(X, y[:, i]) for i in range(6)]

def autoreg_predict(models, history, dates, macro_all=None, net_all=None, fn=None):
    history = history[INDEX_COLS].copy(); preds = []
    for date in dates:
        feats = build_features(history,
                               macro   = macro_all.loc[:date] if macro_all is not None else None,
                               network = net_all.loc[:date]   if net_all   is not None else None)
        row = feats.dropna().iloc[[-1]]
        if fn is not None:
            for c in fn:
                if c not in row.columns: row[c] = 0.0
            row = row[fn]
        y_hat = np.array([m.predict(row.values)[0] for m in models])
        preds.append(y_hat)
        history = pd.concat([history, pd.DataFrame([y_hat], index=[date], columns=INDEX_COLS)])
    return pd.DataFrame(preds, index=dates, columns=INDEX_COLS)

feats_tr = build_features(train, macro_tr, net_tr)
X_tr, y_tr, _ = prepare_xy(feats_tr, train)
fn_tr = list(feats_tr.dropna().columns)
print("Entrenando LightGBM ...")
models_lgbm = train_lgbm(X_tr, y_tr)

# Para el bucle autoregresivo necesitamos acceso a fechas de val también
macro_val = data.get("train_macro_factors")   # contiene las 11956 filas
net_val   = data.get("train_network_metrics")

pred_lgbm_val = autoreg_predict(models_lgbm, train, val.index,
                                macro_all=macro_val, net_all=net_val, fn=fn_tr)
print(f"LightGBM RMSE : {compute_rmse(val, pred_lgbm_val):,.2f}")


Entrenando LightGBM ...
LightGBM RMSE : 3,271.71


## Blend ponderado

In [16]:
from itertools import product

all_preds_val = {"baseline": pred_baseline, "lgbm": pred_lgbm_val}

# Añade predicciones de ARIMA y LSTM si existen en submissions/
import os
for tag, fname in [("arima", "submission_02_arima.csv"),
                   ("lstm",  "submission_04_lstm_seq2seq.csv")]:
    p = os.path.join(SUBMISSIONS, fname)
    if os.path.exists(p):
        all_preds_val[tag] = pd.read_csv(p, parse_dates=[0], index_col=0).reindex(val.index)[INDEX_COLS]
        print(f"  Cargado {tag}: RMSE = {compute_rmse(val, all_preds_val[tag]):,.2f}")

names  = list(all_preds_val.keys())
stacked = np.stack([all_preds_val[n].values for n in names], axis=0)

best_w = np.ones(len(names)) / len(names)
best_r = np.inf

print("Buscando mejores pesos ...")
for combo in product(np.arange(0, 1.01, 0.1), repeat=len(names)):
    w = np.array(combo, dtype=float)
    if w.sum() < 1e-6: continue
    w /= w.sum()
    blended = np.einsum("i,ijk->jk", w, stacked)
    r = compute_rmse(val, pd.DataFrame(blended, index=val.index, columns=INDEX_COLS))
    if r < best_r:
        best_r, best_w = r, w.copy()

print(f"Mejor RMSE ensemble: {best_r:,.2f}")
print(f"Pesos: { {n: round(float(w),3) for n,w in zip(names, best_w)} }")


  Cargado arima: RMSE = nan
  Cargado lstm: RMSE = nan
Buscando mejores pesos ...
Mejor RMSE ensemble: inf
Pesos: {'baseline': 0.25, 'lgbm': 0.25, 'arima': 0.25, 'lstm': 0.25}


## Corrección de Index_D (The Ghost)

In [17]:
from sklearn.linear_model import LinearRegression

source_col, lag, corr_val = find_ghost_source(train, target_col="Index_D", max_lag=30)

def ghost_correct(pred_df, fit_data, last_known_data, ghost="Index_D"):
    """
    fit_data       : datos usados para ajustar la regresión lineal (solo train, sin val).
    last_known_data: datos de los que se extraen los últimos `lag` valores conocidos
                     (train para validación, train_full para test).
    """
    sc, lag, cv = find_ghost_source(fit_data, target_col=ghost, max_lag=30)
    if sc is None or abs(cv) < 0.8:
        print(f"  Correccion saltada (r={cv:.3f})")
        return pred_df
    src = fit_data[sc].values
    tgt = fit_data[ghost].values
    n   = len(src)
    if lag == 0:
        X_fit, y_fit = src.reshape(-1, 1), tgt
    else:
        X_fit, y_fit = src[:n-lag].reshape(-1, 1), tgt[lag:]
    lr = LinearRegression().fit(X_fit, y_fit)
    pred_c = pred_df.copy()
    if lag == 0:
        src_pred = pred_df[sc].values
    else:
        last_known = last_known_data[sc].values[-lag:]
        src_pred   = np.concatenate([last_known, pred_df[sc].values[:-lag]])
    pred_c[ghost] = lr.predict(src_pred.reshape(-1, 1))
    print(f"  Correccion aplicada: {ghost} ~ {sc}  r={cv:.3f}")
    return pred_c


Ghost source: Index_A at lag 3 (corr=0.9999)


## Generar submission final

In [18]:
macro_test   = data.get("test_macro_factors")
network_test = data.get("test_network_metrics")

# Concatenar histórico train + test para que los lags tengan contexto
macro_full_ts = pd.concat([macro,   macro_test])   if (macro   is not None and macro_test   is not None) else macro
net_full_ts   = pd.concat([network, network_test]) if (network is not None and network_test is not None) else network

# Baseline test
pred_bl_test = rolling_mean(train_full, test_dates, 20)

# LightGBM test — reentrenar con train_full
feats_full   = build_features(train_full, macro, network)
X_f, y_f, _  = prepare_xy(feats_full, train_full)
fn_full      = list(feats_full.dropna().columns)
models_f     = train_lgbm(X_f, y_f)
pred_lgbm_test = autoreg_predict(models_f, train_full, test_dates,
                                 macro_all=macro_full_ts, net_all=net_full_ts, fn=fn_full)

test_preds_all = {"baseline": pred_bl_test, "lgbm": pred_lgbm_test}
for tag, fname in [("arima", "submission_02_arima.csv"),
                   ("lstm",  "submission_04_lstm_seq2seq.csv")]:
    p = os.path.join(SUBMISSIONS, fname)
    if os.path.exists(p):
        test_preds_all[tag] = pd.read_csv(p, parse_dates=[0], index_col=0).reindex(test_dates)[INDEX_COLS]

stacked_test = np.stack([test_preds_all[n].values for n in names if n in test_preds_all], axis=0)
w_use = np.array([best_w[i] for i, n in enumerate(names) if n in test_preds_all])
w_use /= w_use.sum()
blended_test = np.einsum("i,ijk->jk", w_use, stacked_test)
ensemble_test = pd.DataFrame(blended_test, index=test_dates, columns=INDEX_COLS)

# fit_data=train (sin val), last_known_data=train_full (para extraer últimos lag valores)
ensemble_test = ghost_correct(ensemble_test, fit_data=train, last_known_data=train_full)
make_submission(ensemble_test, "submission_05_ensemble.csv")
ensemble_test.head()


Ghost source: Index_A at lag 3 (corr=0.9999)
  Correccion aplicada: Index_D ~ Index_A  r=1.000
Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_05_ensemble.csv


,Index_A,Index_B,Index_C,Index_D,Index_E,Index_F
Date,,,,,,
2024-01-01,16093.490915,4626.931325,20.300394,16909.532822,125.076783,40371.966079
2024-01-02,16088.064484,4634.034775,20.371477,16901.201728,125.042887,40846.657300
2024-01-03,16108.394421,4641.264318,20.449395,16828.651868,125.255308,41142.882097
2024-01-04,16124.352515,4643.804997,20.510526,16096.123790,125.404370,41321.260321
2024-01-05,16135.408126,4645.860271,20.559712,16090.696697,125.518401,41469.358041
